# 09 — Inferensi SVLM (Kaggle) — Skripsi Alif

Notebook ini menjalankan **satu model** pada **satu sampel** dan menyimpan hasilnya sebagai
**satu run** dengan identitas unik. Run tidak pernah menimpa run lain: setiap eksekusi
membuat folder baru `runs/<RUN_ID>/` berisi:

| Berkas | Isi |
|---|---|
| `manifest.json` | identitas run: model + revisi (commit Hugging Face), versi pustaka, GPU, sidik prompt dan konfigurasi, sampel, waktu, status |
| `mentah.jsonl` | keluaran mentah model, satu baris per bangunan, ditulis **segera** setelah tiap bangunan selesai |
| `hasil_terurai.csv` | keluaran yang sudah diurai parser 05 |
| `ringkasan.txt` | ringkasan cepat (tingkat urai, penghindaran, matriks kebingungan) |
| `log_berjalan.txt` | jejak per bangunan: uid, tile, chip, label xBD, jawaban, waktu |
| `waktu_komputasi.json` | waktu muat model, statistik waktu per panggilan & per bangunan (rerata, median, sd, min, p95, maks, total), jam dinding, throughput, memori GPU |
| `paket/` | salinan konfigurasi, prompt, dan parser yang dipakai run ini |

**Desain dua panggilan per bangunan** (lihat `desain_inferensi` pada konfigurasi):

1. **Panggilan fitur** — citra pra + pasca (dua citra terpisah, 448 px) → 4 fitur + deskripsi
2. **Panggilan kelas** — citra pasca saja (448 px) → kelas kerusakan + keyakinan

Kedua panggilan tidak saling melihat jawaban, sehingga hubungan fitur–kelas pada Bab 4
bukan artefak prompt.

### Prasyarat Kaggle
1. **Accelerator** = `GPU T4 x2` (atau P100). 2. **Internet** = `On`.
3. **Add Input**: dataset **chip** (`kaggle_chip_400.zip`) dan dataset **paket**
   (`kaggle_paket_eksperimen_v4_<sidik>.zip`).

### Cara menjalankan
- **Pilot 40 chip**: isi sel 2 (`MODEL`, `SAMPEL="kecil"`), jalankan semua sel secara interaktif.
- **Run penuh 400 chip**: pakai **Save Version → Save & Run All (Commit)**. Kaggle menjalankannya
  di latar belakang sampai selesai walau browser ditutup, dan isi `/kaggle/working` tersimpan
  sebagai *Output* versi itu. Satu model ± 80–100 menit.
- Setelah selesai, unduh `run_<RUN_ID>.zip` dari tab Output lalu ekstrak ke
  `KOMPONEN_5\runs\` di laptop. Jangan ganti nama foldernya.

## Sel 1 — periksa GPU dan internet, pasang pustaka

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q "transformers==4.51.3" accelerate timm einops sentencepiece
import sys, torch, transformers, urllib.request
print("python", sys.version.split()[0], "| torch", torch.__version__,
      "| transformers", transformers.__version__, "| cuda", torch.cuda.is_available())
try:
    urllib.request.urlopen("https://huggingface.co", timeout=10); print("internet: AKTIF")
except Exception as e:
    print("internet: MATI ->", type(e).__name__,
          "\n>>> Nyalakan Internet di Notebook options, lalu ulangi sel ini.")
assert torch.cuda.is_available(), "GPU tidak aktif. Notebook options > Accelerator > GPU T4 x2"

## Sel 2 — PARAMETER RUN (satu-satunya sel yang perlu diubah)

| Parameter | Pilihan | Keterangan |
|---|---|---|
| `MODEL` | `"qwen2.5-vl-3b"` / `"internvl3-2b"` | satu model per run |
| `SAMPEL` | `"kecil"` (40) / `"besar"` (400) | pilot dulu dengan `"kecil"` |
| `BATAS` | `None` atau angka | mis. `3` untuk uji cepat; run uji cepat ditandai di manifest |
| `CATATAN` | teks bebas | tujuan run, misal "pilot v4" |
| `LANJUTKAN_DARI` | `None` atau path `mentah.jsonl` | untuk melanjutkan run yang terputus (lihat sel 8) |

In [ ]:
MODEL = "qwen2.5-vl-3b"        # "qwen2.5-vl-3b" atau "internvl3-2b"
SAMPEL = "kecil"               # "kecil" (40, pilot) atau "besar" (400)
BATAS = None                   # None = semua; angka = uji cepat
CATATAN = "pilot prompt v4"
LANJUTKAN_DARI = None          # contoh: "/kaggle/input/run-terputus/mentah.jsonl"
BENIH = 42

REPO = {"qwen2.5-vl-3b": "Qwen/Qwen2.5-VL-3B-Instruct",
        "internvl3-2b": "OpenGVLab/InternVL3-2B"}
assert MODEL in REPO, f"MODEL harus salah satu dari {list(REPO)}"
assert SAMPEL in ("kecil", "besar")

## Sel 3 — temukan dataset, muat konfigurasi, prompt, parser, dan sampel

In [ ]:
import json, hashlib, importlib.util
from pathlib import Path
import pandas as pd

INPUT, KERJA = Path("/kaggle/input"), Path("/kaggle/working")

def cari_satu(pola):
    hasil = sorted(INPUT.rglob(pola))
    if not hasil:
        raise FileNotFoundError(f"'{pola}' tidak ditemukan di /kaggle/input. Sudah Add Input?")
    return hasil[-1]

def sha(path_or_text, n=12):
    b = path_or_text.read_bytes() if isinstance(path_or_text, Path) else path_or_text.encode("utf-8")
    return hashlib.sha256(b).hexdigest()[:n]

F_KONFIG = cari_satu("konfigurasi_fitur.json")
F_PARSER = cari_satu("05_parser_keluaran.py")
F_MANIFES = cari_satu("manifes_v*_*.json")
KONFIG = json.loads(F_KONFIG.read_text(encoding="utf-8"))
MP = json.loads(F_MANIFES.read_text(encoding="utf-8"))
assert MP["fitur_terpilih"] == KONFIG["seleksi"]["terpilih"], \
    "manifes prompt tidak cocok dengan konfigurasi: unggah paket eksperimen yang benar"
PROMPT = {j: cari_satu(MP["prompt"][j]["berkas"]).read_text(encoding="utf-8")
          for j in ("fitur", "kelas")}
for j, teks in PROMPT.items():
    assert sha(teks) == MP["prompt"][j]["sha256_12"], f"prompt {j} berubah dari manifesnya"

spec = importlib.util.spec_from_file_location("parser05", F_PARSER)
P = importlib.util.module_from_spec(spec); spec.loader.exec_module(P)

F_SAMPEL = cari_satu(f"sampel_{SAMPEL}.csv")
sampel = pd.read_csv(F_SAMPEL)
peta_chip = {f.name: f for f in INPUT.rglob("*.png") if f.parent.name in ("pra", "pasca")}
sampel["f_pra"] = sampel["chip_pra"].map(peta_chip)
sampel["f_pasca"] = sampel["chip_pasca"].map(peta_chip)
kurang = sampel[sampel["f_pra"].isna() | sampel["f_pasca"].isna()]
assert kurang.empty, f"{len(kurang)} chip tidak ditemukan, contoh uid {kurang['uid'].head(3).tolist()}"
if BATAS:
    sampel = sampel.groupby("kelas_xbd", group_keys=False).head(max(1, BATAS // 3)).head(BATAS)

print(f"sidik fitur   : {MP['sidik_fitur']}  | prompt v{MP['versi_prompt']}"
      f"  | status seleksi: {KONFIG['seleksi']['status']}")
print(f"fitur terpilih: {KONFIG['seleksi']['terpilih']}")
print(f"sampel        : {SAMPEL}  n={len(sampel)}  {sampel['kelas_xbd'].value_counts().to_dict()}")

## Sel 4 — buat identitas run dan manifest

`RUN_ID` = waktu mulai (UTC) + model + sampel + versi prompt + sidik fitur. Karena memuat
waktu, dua eksekusi dengan pengaturan sama tetap menghasilkan dua run berbeda, dan
keduanya dapat dibandingkan (misalnya untuk memeriksa keterulangan).

In [ ]:
import datetime, platform, torch, transformers

MULAI = datetime.datetime.now(datetime.timezone.utc)
KODE_MODEL = {"qwen2.5-vl-3b": "qwen3b", "internvl3-2b": "ivl2b"}[MODEL]
RUN_ID = (f"{MULAI:%Y%m%d-%H%M%S}_{KODE_MODEL}_{SAMPEL}{BATAS or ''}"
          f"_v{MP['versi_prompt']}_{MP['sidik_fitur']}")
DIR = KERJA / "runs" / RUN_ID
F_MENTAH = DIR / "mentah.jsonl"

gpu = torch.cuda.get_device_properties(0)
MANIFEST = {
    "run_id": RUN_ID,
    "status": "berjalan",
    "waktu_mulai_utc": MULAI.isoformat(timespec="seconds"),
    "catatan": CATATAN,
    "uji_cepat": bool(BATAS),
    "model": {"nama": MODEL, "repo": REPO[MODEL], "revisi": None, "dtype": None},
    "lingkungan": {"platform": "kaggle", "python": platform.python_version(),
                   "torch": torch.__version__, "transformers": transformers.__version__,
                   "gpu": gpu.name, "jumlah_gpu": torch.cuda.device_count()},
    "eksperimen": {"sidik_fitur": MP["sidik_fitur"], "versi_prompt": MP["versi_prompt"],
                   "versi_konfigurasi": KONFIG.get("_versi_konfigurasi"),
                   "status_seleksi": KONFIG["seleksi"]["status"],
                   "fitur_terpilih": KONFIG["seleksi"]["terpilih"],
                   "sha_konfigurasi": sha(F_KONFIG), "sha_parser": sha(F_PARSER),
                   "sha_prompt_fitur": MP["prompt"]["fitur"]["sha256_12"],
                   "sha_prompt_kelas": MP["prompt"]["kelas"]["sha256_12"],
                   "desain": "dua panggilan: fitur(pra+pasca) dan kelas(pasca)"},
    "sampel": {"nama": SAMPEL, "n": int(len(sampel)), "sha_csv": sha(F_SAMPEL),
               "komposisi": sampel["kelas_xbd"].value_counts().to_dict()},
    "dekode": {"do_sample": False, "max_new_tokens": {"fitur": 256, "kelas": 64},
               "benih": BENIH, "ukuran_citra": 448},
    "lanjutan_dari": LANJUTKAN_DARI,
    "catatan_determinisme": ("Dekode greedy (do_sample=False). Operasi GPU tetap dapat "
                             "memberi selisih kecil antar-perangkat atau antar-versi pustaka; "
                             "karena itu versi dan GPU dicatat di atas."),
}
assert not DIR.exists(), f"{DIR} sudah ada; run tidak boleh ditimpa"
DIR.mkdir(parents=True)
def simpan_manifest():
    (DIR / "manifest.json").write_text(json.dumps(MANIFEST, ensure_ascii=False, indent=2),
                                       encoding="utf-8")
simpan_manifest()
# salinan konfigurasi, prompt, dan parser yang dipakai run ini, supaya run tetap dapat
# diurai ulang dengan aturan yang sama walaupun konfigurasi di laptop sudah berubah
import shutil
(DIR / "paket").mkdir()
for f in (F_KONFIG, F_PARSER, F_MANIFES, cari_satu(MP["prompt"]["fitur"]["berkas"]),
          cari_satu(MP["prompt"]["kelas"]["berkas"])):
    shutil.copy(f, DIR / "paket" / f.name)
print("RUN_ID :", RUN_ID)
print("folder :", DIR)

## Sel 5 — muat model

In [ ]:
import time, torch, torchvision.transforms as T
from PIL import Image
torch.manual_seed(BENIH)
T_MUAT = time.time()

if MODEL.startswith("qwen"):
    from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
    try:
        mdl = Qwen2_5_VLForConditionalGeneration.from_pretrained(
            REPO[MODEL], torch_dtype=torch.float16, device_map={"": 0}).eval()
    except TypeError:
        mdl = Qwen2_5_VLForConditionalGeneration.from_pretrained(
            REPO[MODEL], dtype=torch.float16, device_map={"": 0}).eval()
    proc = AutoProcessor.from_pretrained(REPO[MODEL])
    DTYPE = "float16"

    def tanya(gambar, teks, maks_token):
        isi = [{"type": "image", "image": g} for g in gambar] + [{"type": "text", "text": teks}]
        chat = proc.apply_chat_template([{"role": "user", "content": isi}],
                                        tokenize=False, add_generation_prompt=True)
        masuk = proc(text=[chat], images=gambar, return_tensors="pt", padding=True).to(mdl.device)
        with torch.inference_mode():
            keluar = mdl.generate(**masuk, max_new_tokens=maks_token, do_sample=False)
        return proc.decode(keluar[0][masuk["input_ids"].shape[1]:], skip_special_tokens=True)
else:
    from torchvision.transforms.functional import InterpolationMode
    from transformers import AutoModel, AutoTokenizer
    TF = T.Compose([T.Lambda(lambda im: im.convert("RGB")),
                    T.Resize((448, 448), interpolation=InterpolationMode.BICUBIC),
                    T.ToTensor(), T.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))])
    def _muat():
        return AutoModel.from_pretrained(REPO[MODEL], torch_dtype=torch.bfloat16,
                                         low_cpu_mem_usage=True, use_flash_attn=False,
                                         trust_remote_code=True).eval().cuda()
    try:
        mdl = _muat()
    except AttributeError as e:
        if "all_tied_weights_keys" not in str(e):
            raise
        transformers.modeling_utils.PreTrainedModel.all_tied_weights_keys = property(
            lambda self: list(getattr(self, "_tied_weights_keys", None) or []))
        mdl = _muat()
    tok = AutoTokenizer.from_pretrained(REPO[MODEL], trust_remote_code=True, use_fast=False)
    DTYPE = "bfloat16"

    def tanya(gambar, teks, maks_token):
        # Chip berukuran persis 448 x 448, jadi setiap citra = satu petak 448 tanpa
        # pemecahan (dynamic tiling tidak memecah citra persegi seukuran petak).
        px = [TF(g).unsqueeze(0).to(torch.bfloat16).cuda() for g in gambar]
        if len(px) == 1:
            awalan = "<image>\n"
        else:
            awalan = "".join(f"Citra-{i}: <image>\n" for i in range(1, len(px) + 1))
        return mdl.chat(tok, torch.cat(px, 0), awalan + teks,
                        dict(max_new_tokens=maks_token, do_sample=False),
                        num_patches_list=[p.size(0) for p in px])

# revisi (commit) bobot model yang benar-benar terunduh, untuk keterulangan
try:
    from huggingface_hub import HfApi
    REVISI = HfApi().model_info(REPO[MODEL]).sha
except Exception:
    REVISI = getattr(getattr(mdl, "config", None), "_commit_hash", None)
torch.cuda.synchronize()
MANIFEST["model"].update({"revisi": REVISI, "dtype": DTYPE})
MANIFEST["waktu_komputasi"] = {
    "detik_muat_model": round(time.time() - T_MUAT, 1),
    "memori_gpu_setelah_muat_gb": round(torch.cuda.memory_allocated() / 1e9, 2),
}
simpan_manifest()
print(MODEL, "siap | revisi", REVISI, "| dtype", DTYPE,
      "| muat", MANIFEST["waktu_komputasi"]["detik_muat_model"], "detik",
      "| memori GPU", MANIFEST["waktu_komputasi"]["memori_gpu_setelah_muat_gb"], "GB")

## Sel 6 — uji satu bangunan dulu

Sebelum loop panjang, satu bangunan dijalankan dan hasil urainya ditampilkan.
Kalau di sini sudah gagal urai, hentikan dan kirim keluarannya ke peneliti.

In [ ]:
import time
from PIL import Image
def buka(f):
    return Image.open(f).convert("RGB")

r0 = sampel.iloc[0]
t0 = time.time()
m_f = tanya([buka(r0.f_pra), buka(r0.f_pasca)], PROMPT["fitur"], 256)
m_k = tanya([buka(r0.f_pasca)], PROMPT["kelas"], 64)
print(f"uid={r0.uid} | {r0.tile} | chip={r0.chip_pra},{r0.chip_pasca}")
print(f"{time.time()-t0:.1f} detik | label xBD: {r0.kelas_xbd}")
print("--- mentah fitur ---\n", m_f)
print("--- mentah kelas ---\n", m_k)
print("--- terurai ---")
print(P.urai_fitur(m_f, KONFIG))
print(P.urai_kelas(m_k, KONFIG))

## Sel 7 — loop utama

Setiap bangunan langsung ditulis ke `mentah.jsonl` (dengan `flush`), sehingga bila sesi
terputus, bangunan yang sudah selesai tidak hilang. Menjalankan ulang sel ini di sesi yang
sama akan **melewati** bangunan yang sudah ada di berkas, bukan mengulangnya.

**Jejak per bangunan.** Setiap bangunan yang selesai dicetak satu baris: urutan, uid,
tile, nama berkas chip, label xBD, jawaban kelas dan fitur hasil urai cepat, dan waktu.
Baris yang sama disimpan ke `log_berjalan.txt` di folder run, sehingga proses eksekusi
terdokumentasi walaupun keluaran notebook tidak disimpan.

In [ ]:
import traceback

selesai = set()
if F_MENTAH.exists():
    selesai = {json.loads(b)["uid"] for b in F_MENTAH.read_text(encoding="utf-8").splitlines() if b.strip()}
if LANJUTKAN_DARI and not selesai:
    lama = [json.loads(b) for b in Path(LANJUTKAN_DARI).read_text(encoding="utf-8").splitlines() if b.strip()]
    with F_MENTAH.open("w", encoding="utf-8") as fh:
        for b in lama:
            b["dari_run_sebelumnya"] = True
            fh.write(json.dumps(b, ensure_ascii=False) + "\n")
    selesai = {b["uid"] for b in lama}
    print(f"melanjutkan: {len(selesai)} bangunan disalin dari {LANJUTKAN_DARI}")

sisa = sampel[~sampel["uid"].isin(selesai)]
F_LOG = DIR / "log_berjalan.txt"
SINGKAT = {"keutuhan_atap": "atap", "puing_sekitar": "puing",
           "wujud_bangunan": "wujud", "sedimen_lumpur": "sedimen"}

def catat(teks):
    print(teks)
    with F_LOG.open("a", encoding="utf-8") as lg:
        lg.write(teks + "\n")

catat(f"=== {RUN_ID} | mulai {datetime.datetime.now(datetime.timezone.utc):%Y-%m-%d %H:%M:%S} UTC"
      f" | sudah {len(selesai)} | sisa {len(sisa)} ===")
t_awal = time.time()
T_LOOP_MULAI = datetime.datetime.now(datetime.timezone.utc)
torch.cuda.reset_peak_memory_stats()
with F_MENTAH.open("a", encoding="utf-8") as fh:
    for i, r in enumerate(sisa.itertuples(index=False), 1):
        baris = {"uid": r.uid, "tile": r.tile, "kelas_xbd": r.kelas_xbd, "model": MODEL,
                 "run_id": RUN_ID}
        for jenis, gambar, maks in (("fitur", [r.f_pra, r.f_pasca], 256),
                                    ("kelas", [r.f_pasca], 64)):
            t0 = time.time()
            try:
                baris[f"mentah_{jenis}"] = tanya([buka(g) for g in gambar], PROMPT[jenis], maks)
                baris[f"galat_{jenis}"] = None
            except Exception:
                baris[f"mentah_{jenis}"] = None
                baris[f"galat_{jenis}"] = traceback.format_exc(limit=2)
                torch.cuda.empty_cache()
            baris[f"detik_{jenis}"] = round(time.time() - t0, 2)
        baris["detik_total"] = round(baris["detik_fitur"] + baris["detik_kelas"], 2)
        baris["selesai_utc"] = datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")
        fh.write(json.dumps(baris, ensure_ascii=False) + "\n"); fh.flush()
        # urai cepat hanya untuk ditampilkan; hasil resmi tetap dari sel 8 / skrip 10
        hf = P.urai_fitur(baris.get("mentah_fitur") or "", KONFIG)
        hk = P.urai_kelas(baris.get("mentah_kelas") or "", KONFIG)
        fitur_teks = ", ".join(f"{SINGKAT.get(k, k)}={hf.get('fitur_' + k)}"
                               for k in KONFIG["seleksi"]["terpilih"])
        galat = " | GALAT" if (baris["galat_fitur"] or baris["galat_kelas"]) else ""
        catat(f"[{len(selesai) + i:>4}/{len(sampel)}] uid={r.uid} | {r.tile} | "
              f"chip={r.chip_pra},{r.chip_pasca} | xBD={r.kelas_xbd} | "
              f"kelas={hk.get('kelas_kerusakan')} | {fitur_teks} | "
              f"{baris['detik_fitur'] + baris['detik_kelas']:.1f}s{galat}")
        if i % 10 == 0 or i == len(sisa):
            laju = (time.time() - t_awal) / i
            catat(f"       ... {i}/{len(sisa)} selesai | {laju:4.1f} s/bangunan | "
                  f"perkiraan sisa {laju*(len(sisa)-i)/60:5.1f} menit")
T_LOOP_SELESAI = datetime.datetime.now(datetime.timezone.utc)
catat(f"=== loop selesai {T_LOOP_SELESAI:%Y-%m-%d %H:%M:%S} UTC | jam dinding loop "
      f"{(T_LOOP_SELESAI - T_LOOP_MULAI).total_seconds() / 60:.1f} menit ===")

## Sel 8 — urai, ringkas, tutup run

Hasil urai disimpan ke `hasil_terurai.csv`. Ringkasan di sini hanya **pemeriksaan cepat**;
analisis formal (metrik, uji statistik) dilakukan di laptop dengan skrip analisis.

*Bila sesi terputus sebelum sel 7 selesai:* unduh `mentah.jsonl` dari Output, unggah sebagai
dataset, isi `LANJUTKAN_DARI` di sel 2 dengan path-nya, lalu jalankan ulang. Run baru akan
mencatat `lanjutan_dari` di manifest-nya.

In [ ]:
import shutil
from collections import Counter

mentah = [json.loads(b) for b in F_MENTAH.read_text(encoding="utf-8").splitlines() if b.strip()]
baris = []
for m in mentah:
    h = {"uid": m["uid"], "tile": m["tile"], "kelas_xbd": m["kelas_xbd"], "model": m["model"],
         "run_id": RUN_ID, "detik_fitur": m.get("detik_fitur"), "detik_kelas": m.get("detik_kelas")}
    h.update(P.urai_fitur(m.get("mentah_fitur") or "", KONFIG))
    h.update(P.urai_kelas(m.get("mentah_kelas") or "", KONFIG))
    baris.append(h)
df = pd.DataFrame(baris)
df["label_kelas"] = df["kelas_xbd"].map(KONFIG["kelas"]["pemetaan_xbd"])
df.to_csv(DIR / "hasil_terurai.csv", index=False, encoding="utf-8-sig")

L = []
n = len(df)
L.append(f"RUN {RUN_ID}")
L.append(f"model {MODEL} | sampel {SAMPEL} n={n} | prompt v{MP['versi_prompt']} sidik {MP['sidik_fitur']}")
L.append(f"galat inferensi   : fitur {sum(1 for m in mentah if m.get('galat_fitur'))} | "
         f"kelas {sum(1 for m in mentah if m.get('galat_kelas'))}")
L.append(f"fitur sah penuh   : {df['f_sah'].sum()}/{n}")
L.append(f"kelas terurai sah : {df['k_sah'].sum()}/{n}")
L.append(f"kelas tidak_jelas : {df['kelas_tak_teramati'].sum()}/{n}")
L.append("tingkat perbaikan fitur: " + str(Counter(df['f_tingkat_perbaikan'])))
L.append("tingkat perbaikan kelas: " + str(Counter(df['k_tingkat_perbaikan'])))
urut = KONFIG["kelas"]["enum"] + [KONFIG["kelas"]["kode_tak_teramati"]]
mk = pd.crosstab(df["label_kelas"], df["kelas_kerusakan"].fillna("GAGAL"),
                 rownames=["label xBD"], colnames=["prediksi"])
mk = mk.reindex(index=KONFIG["kelas"]["enum"],
                columns=[c for c in urut + ["GAGAL"] if c in mk.columns], fill_value=0)
L.append("\nMatriks kebingungan (baris = label xBD):\n" + mk.to_string())
sah = df[df["kelas_kerusakan"].isin(KONFIG["kelas"]["enum"])]
if len(sah):
    L.append(f"\nakurasi 3 kelas (hanya jawaban sah, n={len(sah)}): "
             f"{(sah['kelas_kerusakan'] == sah['label_kelas']).mean():.3f}")
for k in KONFIG["seleksi"]["terpilih"]:
    L.append(f"\nfitur {k}:\n" + pd.crosstab(df["label_kelas"], df[f"fitur_{k}"].fillna("GAGAL")).to_string())
# ---------------- waktu komputasi ----------------
# Yang diukur: waktu satu panggilan model (praproses citra + generate + decode), per
# jenis panggilan. Tidak termasuk memuat model (dicatat terpisah) dan menulis berkas.
import numpy as np
def statistik(nilai):
    v = np.array([x for x in nilai if x is not None], dtype=float)
    if not len(v):
        return {}
    return {"n": int(len(v)), "total_detik": round(float(v.sum()), 1),
            "rerata": round(float(v.mean()), 2), "median": round(float(np.median(v)), 2),
            "sd": round(float(v.std(ddof=1)), 2) if len(v) > 1 else 0.0,
            "min": round(float(v.min()), 2), "p95": round(float(np.percentile(v, 95)), 2),
            "maks": round(float(v.max()), 2)}
sah_waktu = [m for m in mentah if not (m.get("galat_fitur") or m.get("galat_kelas"))
             and not m.get("dari_run_sebelumnya")]
WAKTU = MANIFEST.get("waktu_komputasi", {})
WAKTU.update({
    "satuan": "detik",
    "cakupan": "per panggilan model: praproses citra + generate + decode; tanpa muat model",
    "per_panggilan_fitur": statistik([m.get("detik_fitur") for m in sah_waktu]),
    "per_panggilan_kelas": statistik([m.get("detik_kelas") for m in sah_waktu]),
    "per_bangunan": statistik([m.get("detik_fitur", 0) + m.get("detik_kelas", 0) for m in sah_waktu]),
    "per_bangunan_tanpa_pertama": statistik([m.get("detik_fitur", 0) + m.get("detik_kelas", 0)
                                             for m in sah_waktu[1:]]),
    "jam_dinding_loop_menit": round((T_LOOP_SELESAI - T_LOOP_MULAI).total_seconds() / 60, 1),
    "memori_gpu_puncak_inferensi_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2),
    "gpu": torch.cuda.get_device_name(0),
})
pb = WAKTU["per_bangunan"]
if pb:
    WAKTU["throughput_bangunan_per_jam"] = round(3600 / pb["rerata"], 1)
MANIFEST["waktu_komputasi"] = WAKTU
(DIR / "waktu_komputasi.json").write_text(json.dumps(WAKTU, ensure_ascii=False, indent=2),
                                          encoding="utf-8")
L.append("\nWAKTU KOMPUTASI (detik)")
L.append(f"  muat model            : {WAKTU.get('detik_muat_model')} s | GPU {WAKTU['gpu']}")
for kunci, nama in (("per_panggilan_fitur", "panggilan fitur "), ("per_panggilan_kelas", "panggilan kelas "),
                    ("per_bangunan", "per bangunan    ")):
    x = WAKTU[kunci]
    if x:
        L.append(f"  {nama}      : rerata {x['rerata']} | median {x['median']} | sd {x['sd']} | "
                 f"min {x['min']} | p95 {x['p95']} | maks {x['maks']} | total {x['total_detik']} (n={x['n']})")
L.append(f"  jam dinding loop      : {WAKTU['jam_dinding_loop_menit']} menit")
L.append(f"  throughput            : {WAKTU.get('throughput_bangunan_per_jam')} bangunan/jam")
L.append(f"  memori GPU            : setelah muat {WAKTU.get('memori_gpu_setelah_muat_gb')} GB | "
         f"puncak inferensi {WAKTU['memori_gpu_puncak_inferensi_gb']} GB")

teks = "\n".join(L)
print(teks)
(DIR / "ringkasan.txt").write_text(teks, encoding="utf-8")

SELESAI = datetime.datetime.now(datetime.timezone.utc)
MANIFEST.update({"status": "selesai" if len(mentah) >= len(sampel) else "terputus",
                 "waktu_selesai_utc": SELESAI.isoformat(timespec="seconds"),
                 "durasi_menit": round((SELESAI - MULAI).total_seconds() / 60, 1),
                 "n_selesai": len(mentah),
                 "detik_rerata_per_bangunan": pb.get("rerata") if pb else None})
simpan_manifest()
arsip = shutil.make_archive(str(KERJA / f"run_{RUN_ID}"), "zip", root_dir=KERJA / "runs", base_dir=RUN_ID)
print("\narsip untuk diunduh:", arsip)

## Sel 9 — contoh keluaran untuk dibaca manual

Enam bangunan acak (dua per kelas) dengan deskripsinya. Bacalah; angka saja tidak cukup
untuk menilai apakah model benar-benar melihat.

In [ ]:
contoh = pd.concat([g.sample(min(2, len(g)), random_state=BENIH) for _, g in df.groupby("label_kelas")])
kolom = ["uid", "label_kelas", "kelas_kerusakan"] + [f"fitur_{k}" for k in KONFIG["seleksi"]["terpilih"]] + ["deskripsi"]
pd.set_option("display.max_colwidth", 120)
display(contoh[kolom])